# Data Engineering for Over-by-Over Modeling
This notebook loads ball-by-ball first-innings data and creates over-level features for training.

In [2]:
import pandas as pd
import numpy as np

RAW_FILE = 'final_ball_by_ball_first_innings.csv'
PROCESSED_FILE = 'processed_data.csv'

In [3]:
def process_ball_by_ball_data(df):
    print('Aggregating ball-by-ball data to over-by-over...')

    working = df.copy()

    rename_map = {
        'over': 'over_no',
        'runs_batter': 'runs',
        'runs_extras': 'extras',
        'wicket_fallen': 'is_wicket'
    }
    working = working.rename(columns=rename_map)

    required_cols = ['match_id', 'over_no', 'runs', 'extras', 'is_wicket']
    missing = [c for c in required_cols if c not in working.columns]
    if missing:
        raise ValueError(f'Missing required columns: {missing}')

    # Source over is 0-indexed in this dataset; convert to 1..20
    working['over_no'] = pd.to_numeric(working['over_no'], errors='coerce') + 1

    for col in ['runs', 'extras', 'is_wicket']:
        working[col] = pd.to_numeric(working[col], errors='coerce').fillna(0)

    working = working.dropna(subset=['match_id', 'over_no'])

    # Calculate total runs per ball
    working['total_runs_ball'] = working['runs'] + working['extras']

    # Group by match and over to get over-level stats
    over_stats = working.groupby(['match_id', 'over_no']).agg(
        runs_this_over=('total_runs_ball', 'sum'),
        wickets_this_over=('is_wicket', 'sum')
    ).reset_index()

    # Calculate cumulative stats
    over_stats['cumulative_runs'] = over_stats.groupby('match_id')['runs_this_over'].cumsum()
    over_stats['cumulative_wickets'] = over_stats.groupby('match_id')['wickets_this_over'].cumsum()
    over_stats['run_rate'] = over_stats['cumulative_runs'] / over_stats['over_no']
    over_stats['balls_remaining'] = 120 - (over_stats['over_no'] * 6)

    # Get final score for each match (target variable)
    final_scores = over_stats.groupby('match_id')['cumulative_runs'].max().reset_index()
    final_scores.rename(columns={'cumulative_runs': 'final_score'}, inplace=True)

    # Merge target score back to the dataset
    over_stats = pd.merge(over_stats, final_scores, on='match_id', how='left')

    return over_stats.sort_values(['match_id', 'over_no']).reset_index(drop=True)

In [4]:
df_raw = pd.read_csv(RAW_FILE)
print('Raw shape:', df_raw.shape)
print('Columns:', df_raw.columns.tolist())

processed_df = process_ball_by_ball_data(df_raw)
print('Processed shape:', processed_df.shape)
processed_df.head()

Raw shape: (599294, 14)
Columns: ['match_id', 'date', 'venue', 'innings', 'team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'runs_batter', 'runs_extras', 'runs_total', 'wicket_fallen']
Aggregating ball-by-ball data to over-by-over...
Processed shape: (95025, 9)


,match_id,over_no,runs_this_over,wickets_this_over,cumulative_runs,cumulative_wickets,run_rate,balls_remaining,final_score
0,211028,1,4,0,4,0,4.000000,114,179
1,211028,2,2,0,6,0,3.000000,108,179
2,211028,3,14,0,20,0,6.666667,102,179
3,211028,4,9,1,29,1,7.250000,96,179
4,211028,5,14,0,43,1,8.600000,90,179


In [5]:
processed_df.to_csv(PROCESSED_FILE, index=False)
print(f'Saved engineered data to {PROCESSED_FILE}')

Saved engineered data to processed_data.csv
